# Fact-Check Notebook: Assignment 1 AI-Generated Analysis

This notebook verifies every specific claim made in `work/design.md` and `work/profile.md` against the raw CSV.  
Run each cell and note whether the output matches the claim. Add your own notes where you disagree.

In [ ]:
import pandas as pd

MISSING = {"?", "", "None", "Unknown/Invalid", "NULL", "NaN"}

# Load everything as strings so we see the raw values exactly as they appear
df = pd.read_csv("../data/diabetic_data.csv", dtype=str, keep_default_na=False)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

---
## Part 1 claims

In [ ]:
# CLAIM: 101,766 rows and 50 columns
rows, cols = df.shape
print(f"Rows: {rows:,}  (claimed: 101,766)  {'PASS' if rows == 101766 else 'FAIL'}")
print(f"Cols: {cols}    (claimed: 50)         {'PASS' if cols == 50 else 'FAIL'}")

In [ ]:
# CLAIM: encounter_id is unique on all 101,766 rows (primary key)
n_enc = df["encounter_id"].nunique()
print(f"Distinct encounter_id: {n_enc:,}  (claimed: 101,766)  {'PASS' if n_enc == 101766 else 'FAIL'}")

In [ ]:
# CLAIM: patient_nbr has 71,518 distinct values
n_pat = df["patient_nbr"].nunique()
print(f"Distinct patient_nbr: {n_pat:,}  (claimed: 71,518)  {'PASS' if n_pat == 71518 else 'FAIL'}")

# CLAIM: 16,773 patients have more than one encounter
multi = (df.groupby("patient_nbr").size() > 1).sum()
print(f"Patients with >1 encounter: {multi:,}  (claimed: 16,773)  {'PASS' if multi == 16773 else 'FAIL'}")

In [ ]:
# CLAIM: admission_type_id has 8 distinct values
# CLAIM: discharge_disposition_id has 26 distinct values
# CLAIM: admission_source_id has 17 distinct values
for col, claimed in [("admission_type_id", 8), ("discharge_disposition_id", 26), ("admission_source_id", 17)]:
    d = df[col].nunique()
    print(f"{col}: {d} distinct  (claimed: {claimed})  {'PASS' if d == claimed else 'FAIL'}")

In [ ]:
# CLAIM: diag_1/2/3 hold 915 distinct ICD-9 codes across all three columns combined
all_codes = pd.concat([df["diag_1"], df["diag_2"], df["diag_3"]])
all_codes = all_codes[~all_codes.isin(MISSING)]
n_codes = all_codes.nunique()
print(f"Distinct ICD-9 codes: {n_codes:,}  (claimed: 915)  {'PASS' if n_codes == 915 else 'FAIL'}")

In [ ]:
# CLAIM: missing value counts per column
claims = {
    "weight":            98569,
    "max_glu_serum":     96420,
    "A1Cresult":         84748,
    "medical_specialty": 49949,
    "payer_code":        40256,
    "race":               2273,
    "diag_3":             1423,
    "diag_2":              358,
    "diag_1":               21,
    "gender":                3,
}
for col, expected in claims.items():
    actual = df[col].isin(MISSING).sum()
    status = "PASS" if actual == expected else "FAIL"
    print(f"{col:25s}: {actual:6,}  (claimed: {expected:6,})  {status}")

---
## Data-quality claims
These are the more interesting ones — the AI made specific claims about inconsistencies in the data.

In [ ]:
# CLAIM: 123 patients have two different real race values across their encounters
# "real" means we exclude '?' entries
race_valid = df[~df["race"].isin(MISSING)][["patient_nbr", "race"]]
multi_race = race_valid.groupby("patient_nbr")["race"].nunique()
conflicting_race = (multi_race > 1).sum()
print(f"Patients with conflicting race: {conflicting_race:,}  (claimed: 123)  {'PASS' if conflicting_race == 123 else 'FAIL'}")

# Show a sample of conflicting patients
sample_ids = multi_race[multi_race > 1].head(3).index
print("\nSample:")
print(race_valid[race_valid["patient_nbr"].isin(sample_ids)].sort_values("patient_nbr"))

In [ ]:
# CLAIM: 3 patients have two genders
gender_valid = df[~df["gender"].isin(MISSING)][["patient_nbr", "gender"]]
multi_gender = gender_valid.groupby("patient_nbr")["gender"].nunique()
conflicting_gender = (multi_gender > 1).sum()
print(f"Patients with conflicting gender: {conflicting_gender:,}  (claimed: 3)  {'PASS' if conflicting_gender == 3 else 'FAIL'}")

In [ ]:
# CLAIM: age changes for 1,574 patients between visits
# (age belongs to the encounter, not the patient, because of this)
multi_age = df.groupby("patient_nbr")["age"].nunique()
changing_age = (multi_age > 1).sum()
print(f"Patients with changing age: {changing_age:,}  (claimed: 1,574)  {'PASS' if changing_age == 1574 else 'FAIL'}")

In [ ]:
# CLAIM: in 5,690 encounters the same ICD-9 code appears in two diagnosis slots
# This is why (encounter_id, diag_position) is the key, not (encounter_id, icd9_code)
def has_duplicate_diag(row):
    codes = [row["diag_1"], row["diag_2"], row["diag_3"]]
    codes = [c for c in codes if c not in MISSING]
    return len(codes) != len(set(codes))

dup_diag = df.apply(has_duplicate_diag, axis=1).sum()
print(f"Encounters with duplicate diag code: {dup_diag:,}  (claimed: 5,690)  {'PASS' if dup_diag == 5690 else 'FAIL'}")

---
## Part 2 claims: normalization row counts
The design doc predicts exact row counts after splitting into normalized tables.

In [ ]:
# CLAIM: encounter table — 101,766 rows
encounter_rows = len(df)
print(f"encounter rows: {encounter_rows:,}  (claimed: 101,766)  {'PASS' if encounter_rows == 101766 else 'FAIL'}")

# CLAIM: patient table — 71,518 rows
patient_rows = df["patient_nbr"].nunique()
print(f"patient rows:   {patient_rows:,}  (claimed: 71,518)  {'PASS' if patient_rows == 71518 else 'FAIL'}")

# CLAIM: diagnosis table — 915 rows
diagnosis_rows = all_codes.nunique()
print(f"diagnosis rows: {diagnosis_rows:,}  (claimed: 915)  {'PASS' if diagnosis_rows == 915 else 'FAIL'}")

# CLAIM: admission_type — 8, discharge_disposition — 26, admission_source — 17
for col, tbl, claimed in [
    ("admission_type_id",        "admission_type",        8),
    ("discharge_disposition_id", "discharge_disposition", 26),
    ("admission_source_id",      "admission_source",      17),
]:
    r = df[col].nunique()
    print(f"{tbl}: {r}  (claimed: {claimed})  {'PASS' if r == claimed else 'FAIL'}")

In [ ]:
# CLAIM: encounter_diagnosis — 303,496 rows
# Logic: 101,766 encounters × 3 slots = 305,298 total slots,
# minus 1,802 slots that are '?' = 303,496 non-missing diagnosis rows

total_slots = len(df) * 3
missing_slots = (
    df["diag_1"].isin(MISSING).sum() +
    df["diag_2"].isin(MISSING).sum() +
    df["diag_3"].isin(MISSING).sum()
)
ed_rows = total_slots - missing_slots

print(f"Total slots: {total_slots:,}  (claimed: 305,298)  {'PASS' if total_slots == 305298 else 'FAIL'}")
print(f"Missing slots: {missing_slots:,}  (claimed: 1,802)  {'PASS' if missing_slots == 1802 else 'FAIL'}")
print(f"encounter_diagnosis rows: {ed_rows:,}  (claimed: 303,496)  {'PASS' if ed_rows == 303496 else 'FAIL'}")

---
## Your own exploration
Use the cells below to investigate anything that failed above, or anything you want to understand better.

In [ ]:
# What are all the distinct race values, including missing markers?
print(df["race"].value_counts(dropna=False))

In [ ]:
# What does a patient with changing age look like?
changing_patients = multi_age[multi_age > 1].index[:3]
df[df["patient_nbr"].isin(changing_patients)][["encounter_id", "patient_nbr", "age"]].sort_values("patient_nbr")

In [ ]:
# Scratch cell — add anything you want to check
